# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Examples on Machine Learning: Alternating Least Squares (ALS)** </center>
---
**Profesor**: Pablo Camarillo Ramirez\
**Estudiante**: Sebastian Tadeo Quiroz Tejeda

# Create SparkSession

In [1]:
from pcamarillor.spark_utils import SparkUtils

su = SparkUtils("ML: ALS", 
                "spark://spark-master:7077")
su.spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/23 01:19:56 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# Example 1: Songs recommednation

In [2]:
# Sample user-song interaction data
data = [(1, 1, 4),
        (1, 2, 5),
        (1, 5, 5),
        (2, 2, 3),
        (2, 3, 4),
        (2, 4, 3),
        (3, 1, 2),
        (3, 3, 5),
        (3, 5, 1)]
  
# Define schema for the DataFrame
schema = SparkUtils.generate_schema([("user_id", "int"), ("song_id", "int"), ("rating", "int")])

# Create DataFrame for interactions
interactions_df = su.spark.createDataFrame(data, schema)
interactions_df.show()

+-------+-------+------+
|user_id|song_id|rating|
+-------+-------+------+
|      1|      1|     4|
|      1|      2|     5|
|      1|      5|     5|
|      2|      2|     3|
|      2|      3|     4|
|      2|      4|     3|
|      3|      1|     2|
|      3|      3|     5|
|      3|      5|     1|
+-------+-------+------+



In [3]:
print(f"Number of items o canciones (n):{interactions_df.groupBy('song_id').count().count()}")
print(f"Number of users (m):{interactions_df.groupBy('user_id').count().count()}")

Number of items o canciones (n):5
Number of users (m):3


In [4]:
from pyspark.ml.recommendation import ALS

als = ALS(
    userCol="user_id", 
    itemCol="song_id", 
    ratingCol="rating", 
    maxIter=10, 
    regParam=0.1, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [5]:
model = als.fit(interactions_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


In [6]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=3)

# Show recommendations
user_recommendations.show(truncate=False)

+-------+-----------------------------------------------+
|user_id|recommendations                                |
+-------+-----------------------------------------------+
|1      |[{2, 4.958433}, {5, 4.8551726}, {1, 3.9412909}]|
|2      |[{3, 3.9445536}, {2, 2.970809}, {4, 2.9085002}]|
|3      |[{3, 4.836294}, {4, 3.276947}, {2, 2.4935796}] |
+-------+-----------------------------------------------+



In [7]:
songs = [
    (1, "song a"),
    (2, "song b"),
    (3, "song c"),
    (4, "song d"),
    (5, "song e")]

songs_schema = SparkUtils.generate_schema([("song_id", "int"), ("title", "string")])
songs_df = su.spark.createDataFrame(songs, songs_schema)

In [8]:
from pyspark.sql.functions import explode

# Explode recommendations for easier reading
recommendations = user_recommendations.select("user_id", explode("recommendations").alias("rec"))
recommendations = recommendations.join(songs_df, recommendations.rec.song_id == songs_df.song_id).select("user_id", "title", "rec.rating")

# Show user-song recommendations with titles
recommendations.show(truncate=False)

+-------+------+---------+
|user_id|title |rating   |
+-------+------+---------+
|1      |song b|4.958433 |
|1      |song e|4.8551726|
|1      |song a|3.9412909|
|2      |song c|3.9445536|
|2      |song b|2.970809 |
|2      |song d|2.9085002|
|3      |song c|4.836294 |
|3      |song d|3.276947 |
|3      |song b|2.4935796|
+-------+------+---------+



In [9]:
predictions = model.transform(interactions_df)
predictions.show(truncate=False)

+-------+-------+------+----------+
|user_id|song_id|rating|prediction|
+-------+-------+------+----------+
|1      |1      |4     |3.9412909 |
|1      |2      |5     |4.958433  |
|1      |5      |5     |4.8551726 |
|2      |2      |3     |2.970809  |
|3      |1      |2     |1.9637711 |
|3      |3      |5     |4.836294  |
|3      |5      |1     |1.0455084 |
|2      |3      |4     |3.9445536 |
|2      |4      |3     |2.9085002 |
+-------+-------+------+----------+



In [10]:
# Evaluate the Recommendation System
from pyspark.ml.evaluation import RegressionEvaluator
# Set up evaluator to compute RMSE
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.08732525881616925


# Lab 12: Building a Recommendation System with ALS 

In [11]:
movies_ratings_path = "/opt/spark/work-dir/data/ml/als"

movies_ratings_schema = SparkUtils.generate_schema([("user_id", "int"), ("movie_id", "int"), ("rating", "int"),("timestamp", "int")])

# Source https://github.com/databricks/Spark-The-Definitive-Guide/blob/master/data/sample_movielens_ratings.txt
movies_ratings_df = su.spark.read \
                    .option("header", "false") \
                    .option("delimiter", "::") \
                    .schema(movies_ratings_schema) \
                    .csv(movies_ratings_path)

movies_ratings_df.printSchema()
movies_ratings_df.show(n=3)

root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: integer (nullable = true)

+-------+--------+------+----------+
|user_id|movie_id|rating| timestamp|
+-------+--------+------+----------+
|      0|       2|     3|1424380312|
|      0|       3|     1|1424380312|
|      0|       5|     2|1424380312|
+-------+--------+------+----------+
only showing top 3 rows


## Create & Train the ML Model

In [12]:
print(f"Number of movies (n):{movies_ratings_df.groupBy('movie_id').count().count()}")
print(f"Number of users (m):{movies_ratings_df.groupBy('user_id').count().count()}")

Number of movies (n):100
Number of users (m):30


In [13]:
als = ALS(
    userCol="user_id", 
    itemCol="movie_id", 
    ratingCol="rating", 
    maxIter=22, 
    regParam=0.0001, 
    rank=5, # Controls the dimensionality of the latent vector space for 
            # users and items.
    coldStartStrategy="drop"  # Avoids NaN predictions
)

In [14]:
model = als.fit(movies_ratings_df)
print("Recommendation system generated successfully")

Recommendation system generated successfully


## Persist the model

## Predictions

In [15]:
# Generate recommendations for each user
user_recommendations = model.recommendForAllUsers(numItems=5)

# Show recommendations
user_recommendations.show(truncate=False)

+-------+-------------------------------------------------------------------------------------+
|user_id|recommendations                                                                      |
+-------+-------------------------------------------------------------------------------------+
|0      |[{76, 3.4935741}, {26, 2.850363}, {70, 2.590386}, {41, 2.4904175}, {42, 2.4343755}]  |
|10     |[{26, 5.7741456}, {76, 4.2191257}, {46, 3.1839075}, {93, 2.8621}, {25, 2.831998}]    |
|20     |[{28, 3.6807642}, {77, 3.6175528}, {59, 3.5903673}, {75, 3.5418494}, {22, 3.4608903}]|
|1      |[{77, 3.0851817}, {59, 3.0374844}, {75, 3.0261402}, {28, 2.9350991}, {22, 2.8902805}]|
|11     |[{26, 22.446056}, {46, 11.327373}, {42, 7.641684}, {55, 7.1028795}, {49, 7.0936985}] |
|21     |[{41, 4.9038467}, {53, 4.867147}, {70, 4.7227926}, {29, 4.60326}, {52, 4.453446}]    |
|22     |[{28, 5.807121}, {59, 5.431108}, {77, 5.254656}, {75, 5.2113585}, {51, 5.017703}]    |
|2      |[{83, 5.0336576}, {93, 5.008719

In [16]:
predictions = model.transform(movies_ratings_df)
predictions.show(truncate=False)

+-------+--------+------+----------+----------+
|user_id|movie_id|rating|timestamp |prediction|
+-------+--------+------+----------+----------+
|22     |0       |1     |1424380312|0.9637323 |
|22     |3       |2     |1424380312|1.815712  |
|22     |5       |2     |1424380312|1.6579621 |
|22     |6       |2     |1424380312|2.3292217 |
|22     |9       |1     |1424380312|1.9033283 |
|22     |10      |1     |1424380312|0.94174266|
|22     |11      |1     |1424380312|0.9882666 |
|22     |13      |1     |1424380312|1.5301058 |
|22     |14      |1     |1424380312|1.4836811 |
|22     |16      |1     |1424380312|0.7589704 |
|22     |18      |3     |1424380312|3.1838224 |
|22     |19      |1     |1424380312|1.4267197 |
|22     |22      |5     |1424380312|4.8629804 |
|22     |25      |1     |1424380312|0.83715725|
|22     |26      |1     |1424380312|1.0812359 |
|22     |29      |3     |1424380312|3.4988303 |
|22     |30      |5     |1424380312|4.1280317 |
|22     |32      |4     |1424380312|3.42

## Test ML Model

In [17]:
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# Calculate RMSE
rmse = evaluator.evaluate(predictions)
print(f"Root-mean-square error (RMSE) = {rmse}")

Root-mean-square error (RMSE) = 0.5359123148496227


In [18]:
su.spark.stop()